# Solutions · 11 · Coupled systems and DAEs

Worked solutions to the exercises of [notebook 11](../notebooks/11_coupled_systems_and_daes.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engmolines is missing): install it from GitHub
    import engmolines
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engmolines"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engmolines import problems as pr, integrators as ti, grids, boundary as bc, operators as op
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

Replace the adsorber's linear isotherm by a Langmuir isotherm q* = q_max K c/(1 + K c). Does the front sharpen or spread (a favourable isotherm sharpens it)? Measure the breakthrough curve's width.

In [2]:
n, K, q_max, k_ldf, eps, u_, D = 100, 5.0, 2.0, 5.0, 0.4, 1.0, 1e-3
g = grids.uniform(0, 1, n)
left, right = bc.Robin(1.0, -D / u_, 1.0), bc.Neumann(0.0)
def column(isotherm):
    def rhs(t, y):
        c, q = y[: n + 1], y[n + 1 :]
        dq = k_ldf * (isotherm(c) - q)
        return np.r_[op.advection(c, g, u_, left, right, t, "upwind") + op.diffusion(c, g, D, left, right, t) - (1 - eps) / eps * dq, dq]
    return ti.bdf(rhs, np.zeros(2 * (n + 1)), (0, 15.0), 0.02, 2, band=(n + 2, n + 2), n_out=150)
for name, iso in (("linear q* = K c", lambda c: K * c), ("Langmuir q* = q_max K c/(1 + K c)", lambda c: q_max * K * c / (1 + K * c))):
    s = column(iso)
    c_out = s.y[:, n]
    t10, t90 = np.interp(0.1, c_out, s.t), np.interp(0.9, c_out, s.t)
    print(f"{name:<34}: breakthrough from 10 % to 90 % of the feed takes {t90 - t10:.2f} time units (midpoint at {np.interp(0.5, c_out, s.t):.2f})")

linear q* = K c                   : breakthrough from 10 % to 90 % of the feed takes 5.02 time units (midpoint at 8.37)


Langmuir q* = q_max K c/(1 + K c) : breakthrough from 10 % to 90 % of the feed takes 0.67 time units (midpoint at 3.44)


A favourable (Langmuir) isotherm makes the high concentrations travel faster than the low ones (the
front is self-sharpening and reaches a constant pattern), so the breakthrough curve is steeper than with the
linear isotherm, whose front spreads diffusively. The method of lines handles the nonlinear isotherm by
putting it in the uptake term; nothing else changes.

## Exercise 2

Write the catalyst-pellet problem as a DAE with a film-resistance boundary condition at the surface: 0 = Sh (1 - c_s) - c_r(1). Solve with `dae_bdf` and compare the effectiveness with the no-resistance case.

In [3]:
phi, Sh = 3.0, 5.0
g = grids.radial(1.0, 60, "spherical"); n = g.n; dx = g.dx
def F(t, y, yp):
    c = y
    res = np.empty(n)
    interior = op.diffusion(c, g, 1.0, None, bc.Dirichlet(c[-1]), t, source=lambda x, c, t: -(phi**2) * c)     # the Dirichlet value is c's own last entry: no constraint from the operator
    res[:-1] = yp[:-1] - interior[:-1]
    res[-1] = Sh * (1 - c[-1]) - (3 * c[-1] - 4 * c[-2] + c[-3]) / (2 * dx)        # film resistance: Sh (1 - c_s) = c_r(1)
    return res
y0 = np.zeros(n); y0[-1] = Sh / (Sh + 1)
s = ti.dae_bdf(F, y0, (0, 2.0), 0.005, order=2, band=(2, 2))
c = s.y[-1]
flux = (3 * c[-1] - 4 * c[-2] + c[-3]) / (2 * dx)
eta_film = 3 * flux / phi**2
eta_exact = 3 * (phi / np.tanh(phi) - 1) / phi**2 / (1 + (phi / np.tanh(phi) - 1) / Sh)      # with film resistance
print(f"surface concentration {c[-1]:.4f} (below 1: the film); effectiveness {eta_film:.4f} vs the exact film-resistance value {eta_exact:.4f}; without film {3 * (phi / np.tanh(phi) - 1) / phi**2:.4f}")

surface concentration 0.7129 (below 1: the film); effectiveness 0.4785 vs the exact film-resistance value 0.4787; without film 0.6716


The film condition Sh(1 − c_s) = c_r at the surface is an algebraic equation in the surface unknown; written
as the last row of a DAE it is enforced exactly at every step, and the effectiveness factor drops by the
factor 1/(1 + (φ coth φ − 1)/Sh) of the series resistances - which the DAE solution reproduces.

## Exercise 3

**Implement it yourself:** add a second adsorbing species with a different K to the column and show that the two breakthrough fronts travel at their own stoichiometric speeds.

In [4]:
n, eps, k_ldf, u_, D = 100, 0.4, 5.0, 1.0, 1e-3
Ks = (2.0, 8.0)
g = grids.uniform(0, 1, n)
left, right = bc.Robin(1.0, -D / u_, 1.0), bc.Neumann(0.0)
def rhs(t, y):
    out = []
    for j, K in enumerate(Ks):
        c, q = y[2 * j * (n + 1):(2 * j + 1) * (n + 1)], y[(2 * j + 1) * (n + 1):(2 * j + 2) * (n + 1)]
        dq = k_ldf * (K * c - q)
        out += [op.advection(c, g, u_, left, right, t, "upwind") + op.diffusion(c, g, D, left, right, t) - (1 - eps) / eps * dq, dq]
    return np.concatenate(out)
s = ti.bdf(rhs, np.zeros(4 * (n + 1)), (0, 20.0), 0.02, 2, band=(n + 2, n + 2), n_out=200)
for j, K in enumerate(Ks):
    c_out = s.y[:, 2 * j * (n + 1) + n]
    print(f"species with K = {K}: breakthrough at t = {np.interp(0.5, c_out, s.t):.2f} (stoichiometric 1 + (1-eps)K/eps = {1 + (1 - eps) / eps * K:.2f})")

species with K = 2.0: breakthrough at t = 3.89 (stoichiometric 1 + (1-eps)K/eps = 4.00)
species with K = 8.0: breakthrough at t = 12.85 (stoichiometric 1 + (1-eps)K/eps = 13.00)


Two independently adsorbing species travel at their own stoichiometric speeds, u/(1 + (1 − ε)K/ε), and
break through at different times - the principle of chromatography. With independent isotherms the species
do not interact; the state vector simply stacks both pairs and the band structure repeats.